In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import tarfile

In [6]:
data_path = Path("/home/vivek/Documents/Vivek-projects/aircraft-state-estimation/data/raw/states_2022-01-17-07.csv.tar")


with tarfile.open(data_path, mode="r:*") as archive:
    csv_members = [
        member
        for member in archive.getmembers()
        if member.isfile()
        and member.name.lower().endswith((".csv", ".csv.gz"))
    ]

    if not csv_members:
        raise FileNotFoundError("No CSV or CSV.GZ found inside the archive")

    csv_file = archive.extractfile(csv_members[0])

    if csv_file is None:
        raise RuntimeError("Could not read CSV member")

    compression = (
        "gzip" if csv_members[0].name.lower().endswith(".gz") else None
    )

    df = pd.read_csv(csv_file, compression=compression)

df.head()

,time,icao24,lat,lon,velocity,heading,vertrate,callsign,onground,alert,spi,squawk,baroaltitude,geoaltitude,lastposupdate,lastcontact
0,1642402800,4d22bd,42.908268,4.958235,234.494500,232.577023,0.00000,RYR1BK,False,False,False,4144.0,11887.2,11986.26,1.642403e+09,1.642403e+09
1,1642402800,4d22b9,41.017868,1.681702,232.756775,227.956311,0.00000,RYR74TQ,False,False,False,4044.0,11277.6,11445.24,1.642403e+09,1.642403e+09
2,1642402800,342188,41.092392,2.115295,229.531976,264.340518,0.32512,IBE3239,False,False,False,4045.0,11887.2,12039.60,1.642403e+09,1.642403e+09
3,1642402800,49528e,44.593094,2.223947,224.394906,238.022464,0.32512,TAP551H,False,False,False,614.0,11887.2,11993.88,1.642403e+09,1.642403e+09
4,1642402800,440823,46.586407,11.734909,242.032097,239.892588,-0.32512,AUA403,False,False,False,1000.0,9144.0,9197.34,1.642403e+09,1.642403e+09


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1219248 entries, 0 to 1219247
Data columns (total 16 columns):
 #   Column         Non-Null Count    Dtype  
---  ------         --------------    -----  
 0   time           1219248 non-null  int64  
 1   icao24         1219248 non-null  str    
 2   lat            1087108 non-null  float64
 3   lon            1087108 non-null  float64
 4   velocity       991127 non-null   float64
 5   heading        991127 non-null   float64
 6   vertrate       991525 non-null   float64
 7   callsign       1154862 non-null  str    
 8   onground       1219248 non-null  bool   
 9   alert          1219248 non-null  bool   
 10  spi            1219248 non-null  bool   
 11  squawk         822731 non-null   float64
 12  baroaltitude   1019852 non-null  float64
 13  geoaltitude    935629 non-null   float64
 14  lastposupdate  1087108 non-null  float64
 15  lastcontact    1219248 non-null  float64
dtypes: bool(3), float64(10), int64(1), str(2)
memory usage: 140.3 MB


In [ ]:
state_columns = [
    "x",
    "y",
    "baroaltitude",
    "velocity",
    "heading",
    "vertrate",
]

support_columns = [
    "time",
    "icao24",
    "callsign",
    "onground",
    "lastposupdate",
    "lastcontact",
]

df["position_age"] = df["time"] - df["lastposupdate"]
df["contact_age"] = df["time"] - df["lastcontact"]